In [7]:
import gensim.downloader as api
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import plotly.graph_objects as go

In [2]:
print("Downloading/Loading the model (this might take a minute)...")
model = api.load("glove-wiki-gigaword-50")

Downloading/Loading the model (this might take a minute)...


In [ ]:
def visualize_word_vectors_animated(target_word, topn=8):
    # --- 1. Data Preparation ---
    if target_word not in model:
        print(f"Word '{target_word}' not in vocabulary.")
        return

    similar_words = [word for word, _ in model.most_similar(target_word, topn=topn)]
    words_to_plot = [target_word] + similar_words
    vectors = np.array([model[w] for w in words_to_plot])

    # Reduce dimensions to 3D
    pca = PCA(n_components=3)
    vectors_3d = pca.fit_transform(vectors)

    # --- 2. Build the Plotly Figure ---
    fig = go.Figure()

    for i, word in enumerate(words_to_plot):
        x, y, z = vectors_3d[i]
        is_target = (word == target_word)
        
        color = 'gold' if is_target else 'red'
        line_width = 6 if is_target else 3
        opacity = 1.0 if is_target else 0.6

        # Draw the vector line from origin
        fig.add_trace(go.Scatter3d(
            x=[0, x], y=[0, y], z=[0, z],
            mode='lines+text',
            line=dict(color=color, width=line_width),
            text=["", f"E({word})"],
            textposition="top center",
            textfont=dict(color=color, size=14 if is_target else 11),
            name=word,
            opacity=opacity
        ))

        # Add an arrowhead (cone) at the tip
        fig.add_trace(go.Cone(
            x=[x], y=[y], z=[z],
            u=[x*0.1], v=[y*0.1], w=[z*0.1],
            colorscale=[[0, color], [1, color]],
            showscale=False,
            anchor="tip"
        ))

    # --- 3. Animation & Layout Settings ---
    fig.update_layout(
        template="plotly_dark",
        title=f"3D Vector Space: {target_word}",
        scene=dict(
            xaxis=dict(range=[-1, 1], autorange=True),
            yaxis=dict(range=[-1, 1], autorange=True),
            zaxis=dict(range=[-1, 1], autorange=True),
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        showlegend=False
    )

    # This creates a rotating camera sequence
    x_eye, y_eye, z_eye = 1.25, 1.25, 0.8
    frames = []
    for t in np.arange(0, 6.28, 0.1):  # Full circle
        frames.append(go.Frame(layout=dict(scene_camera=dict(eye=dict(
            x=x_eye * np.cos(t), 
            y=y_eye * np.sin(t), 
            z=z_eye
        )))))

    fig.frames = frames

    fig.update_layout(
        updatemenus=[dict(type="buttons",buttons=[dict(label="Play Animation",method="animate",
                          args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True, mode="immediate", loop=True)])])])

    fig.show()

In [11]:
visualize_word_vectors_animated("phone") # feel free to change the word to visualize other embeddings!